In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Load data
df_fp = pd.read_csv('results_FP.tsv', sep='\t')
df_sg = pd.read_csv('results_SG.tsv', sep='\t')
df_pg = pd.read_csv('results_PG.tsv', sep='\t')

# Preserve original trait order from the first file
trait_order = df_fp['Trait'].tolist()

df_fp['MainTrait'] = 'SFC frontal pole'
df_sg['MainTrait'] = 'SFC supramarginal gyrus'
df_pg['MainTrait'] = 'SFC precentral gyrus'

df_all = pd.concat([df_fp, df_sg, df_pg], ignore_index=True)

# Pivot for correlation values and p-values
corr_matrix = df_all.pivot(index='Trait', columns='MainTrait', values='GeneticCorrelation')
pval_matrix = df_all.pivot(index='Trait', columns='MainTrait', values='Pvalue')

# Reorder rows to match original CSV order
corr_matrix = corr_matrix.reindex(trait_order)
pval_matrix = pval_matrix.reindex(trait_order)

corr_matrix = corr_matrix.fillna(0)
pval_matrix = pval_matrix.fillna(0)

# Create text and color matrix
text_matrix = []
color_matrix = np.array(corr_matrix.values, dtype=float)

for i, row in enumerate(corr_matrix.values):
    text_row = []
    for j, val in enumerate(row):
        pval = pval_matrix.iloc[i, j]
        
        # Check if this is a self-correlation (row and column trait match)
        if corr_matrix.index[i] == corr_matrix.columns[j]:
            text_row.append("")  # Empty text for self-correlations
            color_matrix[i, j] = np.nan  # grey
        else:
            asterisk = '*' if pval < 0.05 else ''
            text_row.append(f"{val:.2f}{asterisk}")
    text_matrix.append(text_row)


fig = go.Figure(data=go.Heatmap(
    z=color_matrix,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='curl',
    zmid=0,
    colorbar=dict(
        title='Genetic Correlation (rg)',
        titleside='right',
        ticks='outside',
        thickness=20,
        len=0.75,
        y=0.5,
        yanchor='middle'
    ),
    text=text_matrix,
    texttemplate="%{text}",
    hoverinfo='text',
    showscale=True
))

fig.update_layout(
    xaxis=dict(title='', side='bottom'),
    yaxis=dict(title='', autorange='reversed'),
    width=680,
    height=20 * len(corr_matrix) + 100,
    margin=dict(l=100, r=50, t=80, b=50),
    font=dict(size=11)
)
#fig.write_image('ldsc.png')
fig.show()